# 21 — E18: does MORE weight make a diffuse value lead? (batch: one formulation per arm, ~1 h each)

Diagnostic requested by Ethan (2026-09-04). Each arm = a block's forward scenario (connectivity → S2,
biodiversity → S3) with that block's weights × `mult`; targets unchanged; SSP585; certified anchor +
guarded MGA (k = 50, g = 5 %, block floors) — the package semantics. **Run All solves every arm in
`ARMS` that has no output yet** (resumable; live internet). Outputs → `runs/e18_<base>x<mult>_ssp585/`.
The ×5 connectivity arm (ran 2026-09-04) fired the pre-registered rule WEIGHT-LIMITED (R10.13); the
remaining arms locate the crossover (×2 = influence share 0.73), replicate both doses on biodiversity, and add the
**carbon weights-only counterfactual** (S4's doubled carbon share at S0's targets — the lever the other three get).


In [1]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd()")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
ctx <- pr_setup(mpath, PROJ)
ctx <- modifyList(ctx, pr_ingest(ctx))
ctx <- modifyList(ctx, pr_planning_units(ctx))
MANI <- read.csv(file.path(PROJ, "analyses/y2y/spec/manifest.csv"), stringsAsFactors = FALSE)
ER <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/e_round_v13.json"))
BLOCKS <- lapply(ER$e17_t3$blocks, unlist)
RUNS <- file.path(PROJ, "analyses/y2y/runs")
BASES <- c(connectivity = "s2_ssp585_theta5", biodiversity = "s3_ssp585_theta5", carbon = "s4_ssp585_theta3")

# the arms (block, multiplier on the block's forward scenario); each skipped when its output exists
ARMS <- list(list(block = "connectivity", mult = 2),      # crossover: influence share 0.73
             list(block = "connectivity", mult = 5),      # ran 2026-09-04 (share 0.83) -- skipped
             list(block = "biodiversity", mult = 2),      # matched design on the other diffuse block
             list(block = "biodiversity", mult = 5),
             list(block = "carbon", mult = 1, targets_from = "s0_ssp585_theta5"))   # carbon-forward WEIGHTS ONLY: S4's doubled
                                                                                    # share at S0's targets = the lever the others get
arm_dir <- function(a) file.path(RUNS, sprintf("e18_%sx%g%s_ssp585", substr(BASES[[a$block]], 1, 2), a$mult,
                                                  if (is.null(a$targets_from)) "" else "_wonly"))
for (a in ARMS) cat(sprintf("arm %-13s x%-4g %-22s -> %s %s\n", a$block, a$mult,
                            if (is.null(a$targets_from)) "" else paste0("targets from ", substr(a$targets_from, 1, 2)),
                            basename(arm_dir(a)), if (file.exists(file.path(arm_dir(a), "mga_guard_g05.tif"))) "(done)" else "(TODO)"))


manifest refreshed from config.py (analysis=y2y)
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
arm connectivity  x2                           -> e18_s2x2_ssp585 (done)
arm connectivity  x5                           -> e18_s2x5_ssp585 (done)
arm biodiversity  x2                           -> e18_s3x2_ssp585 (done)
arm biodiversity  x5                           -> e18_s3x5_ssp585 (done)
arm carbon       

In [2]:
# ---- the batch: certified anchor + guarded MGA per arm (resumable) ---------------------------------
run_arm <- function(block, mult, targets_from = NULL) {
  base <- BASES[[block]]
  OUT <- arm_dir(list(block = block, mult = mult, targets_from = targets_from))
  tif <- file.path(OUT, "mga_guard_g05.tif")
  if (file.exists(tif)) { cat(sprintf("== %s exists -- skipped\n", basename(OUT))); return(invisible(NULL)) }
  dir.create(OUT, showWarnings = FALSE, recursive = TRUE)
  row <- MANI[MANI$formulation_id == base, ]
  w <- jsonlite::fromJSON(row$weight_vector); t <- jsonlite::fromJSON(row$target_vector)
  if (!is.null(targets_from)) t <- jsonlite::fromJSON(MANI[MANI$formulation_id == targets_from, ]$target_vector)
  for (f in BLOCKS[[block]]) w[[f]] <- w[[f]] * mult
  cat(sprintf("\n===== %s: %s weights x%g =====\n", basename(OUT), block, mult)); print(unlist(w))
  actx <- pr_override(ctx, targets = t, feature_weight_multipliers = w,
                      results_dir = file.path("analyses/y2y/runs", basename(OUT)), results_subdir = "build",
                      solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  t0 <- proc.time()[["elapsed"]]
  anchor <- mga_anchor(cm, opt_gap = 1e-4)
  r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r))
  v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
  terra::writeRaster(r, file.path(OUT, "anchor.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255,
                     gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  gen <- mga_generate(cm, anchor, g = 0.05, k = 50, floors = list(ctx = actx, blocks = BLOCKS, g = 0.05))
  layers <- lapply(seq_len(gen$k), function(i) {
    rr <- terra::rast(actx$cost); vv <- rep(NA_integer_, terra::ncell(rr))
    vv[cm$pu_index] <- as.integer(gen$members[i, ]); terra::values(rr) <- vv; rr })
  s <- terra::rast(layers); names(s) <- sprintf("guard_%02d", seq_len(gen$k))
  terra::writeRaster(s, tif, overwrite = TRUE, datatype = "INT1U", NAflag = 255,
                     gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  write.csv(gen$certificates, file.path(OUT, "certificates_guard.csv"), row.names = FALSE)
  jsonlite::write_json(list(experiment = "E18", base = base, block = block, mult = mult, targets_from = targets_from, weight_vector = w,
                            target_vector = t, anchor_objective = anchor$z, anchor_gap = anchor$gap,
                            anchor_runtime_s = anchor$runtime, k = gen$k, g = 0.05, floor_g = 0.05,
                            wall_s = proc.time()[["elapsed"]] - t0, created_utc = format(Sys.time(), tz = "UTC")),
                       file.path(OUT, "e18_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  cat(sprintf("== %s done: anchor %.6f | %d members | %.1f min\n", basename(OUT), anchor$z, gen$k, (proc.time()[["elapsed"]] - t0) / 60))
}
for (a in ARMS) run_arm(a$block, a$mult, a$targets_from)
cat("\nE18 BATCH COMPLETE -- next: analyses/y2y/22_e18_analysis.ipynb\n")


== e18_s2x2_ssp585 exists -- skipped
== e18_s2x5_ssp585 exists -- skipped
== e18_s3x2_ssp585 exists -- skipped
== e18_s3x5_ssp585 exists -- skipped

===== e18_s4x1_wonly_ssp585: carbon weights x1 =====
   climate_type_macrorefugia   transboundary_connectivity 
                    1.228700                     0.563274 
           climate_corridors   irrecoverable_carbon_m_soc 
                    0.985798                     1.165868 
irrecoverable_carbon_biomass           aoh_richness_birds 
                    0.501302                     1.118079 
        aoh_richness_mammals 
                    1.436979 
  override targets          -> irrecoverable_carbon_m_soc=0.332
  override feature_weight_multipliers -> climate_type_macrorefugia=1.2287, transboundary_connectivity=0.563274, climate_corridors=0.985798, irrecoverable_carbon_m_soc=1.16587, irrecoverable_carbon_biomass=0.501302, aoh_richness_birds=1.11808, aoh_richness_mammals=1.43698
  override results_dir      -> analyses/y2y/runs

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.332 and 1)
││└•weights:    continuous values (between 0.025 and 1.436979)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272962 cols (1272914 pu + 48 aux) x 49 rows | 191029 locked pu | modelsense min
anchor: objective 4.897959 (bound 4.897900, gap 1.20e-05) | 381,874 selected | 56 s
band wall appended: obj0 . x <= 5.142857  (g = 0.05 on z* = 4.897959)
  block floor core_habitat   anchor capture 0.4336 -> floor 0.4119
  block floor connectivity   anchor capture 0.5923 -> floor 0.5627
  block floor carbon         anchor capture 0.7405 -> floor 0.7034
  block floor biodiversity   anchor capture 0.6573 -> floor 0.6245
g=0.05 iter 01/50: band 5.142854 (+5.00% of z*) OK | ham(anchor) 345,900 | 67 s
g=0.05 iter 02/50: band 5.142857 (+5.00% of z*) OK | ham(anchor) 267,326 | 67 s
g=0.05 iter 03/50: band 5.142858 (+5.00% of z*) OK | ham(anchor) 217,798 | 74 s
g=0.05 iter 04/50: band 5.142856 (+5.00% of z*) OK | ham(anchor) 220,826 | 71 s
g=0.05 iter 05/50: band 5.142857 (+5.00% of z*) OK | ham(anchor) 232,814 | 74 s
g=0.05 iter 06/50: band 5.142857 (+5.00% of z*) OK | ham(anchor) 232,666 | 67 s
g=0.05 